<a href="https://colab.research.google.com/github/Ascenderr999/devjoint-intern-job/blob/main/Devjoint_Week_3_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Checkpoint 1: Train/Test Bölgüsünün Preprocessing-dən Əvvəl Aparılması

Machine Learning modelində ən böyük risklərdən biri **Data Leakage** (Məlumat Sızması) problemidir. Əgər məlumatların təmizlənməsi, dəyərlərin doldurulması (imputation), standartlaşdırma (scaling) və ya enkoding (encoding) əməliyyatları bütün dataset üzərində bölgüdən əvvəl aparılarsa, test datasının statistik göstəriciləri train datasının daxilinə sızır. Bu halda model real həyatda görmədiyi test datasını dolayısı ilə tanıyır və saxta yüksək dəqiqlik göstərir.

Bu xətanın qarşısını almaq üçün dataset-i istənilən preprocessing (fərziyyə və transformasiya) addımından əvvəl Train (80%) və Test (20%) hissələrinə böldüm.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('turbo.az_cars.csv', on_bad_lines='skip', engine='python')

print("Dataset-dəki sütunlar:", df.columns.tolist())

target_col = [col for col in df.columns if 'price' in col.lower() or 'qiymət' in col.lower() or 'qiymet' in col.lower()][0]
print("İstifadə olunan qiymət sütunu:", target_col)

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train dataseti həcmi:", X_train.shape)
print("Test dataseti həcmi:", X_test.shape)

Dataset-dəki sütunlar: ['id_x', 'car_rel_url_x', 'datetime_scrape', 'name', 'price_x', 'currency_x', 'datetime_product', 'city', 'day', 'hour', 'attributes', 'production_year', 'engine_displacement_num', 'engine_displacement_unit', 'kilometrage_num', 'kilometrage_unit', 'barter', 'loan', 'salon', 'spare_parts', 'vip', 'featured', 'img_url', 'id_y', 'cars_id', 'car_rel_url_y', 'datetime', 'description', 'price_y', 'currency_y', 'owner_name', 'shop_name', 'phone', 'updated', 'views', 'vin', 'car_details_id_x', 'Ban növü', 'Buraxılış ili', 'Hansı bazar üçün yığılıb', 'Marka', 'Model', 'Mühərrik', 'Qəzalı', 'Rəng', 'Sahiblər', 'Sürətlər qutusu', 'Vəziyyəti', 'Yeni', 'Yerlərin sayı', 'Yürüş', 'Ötürücü', 'Şəhər', 'car_details_id_y', 'car_rel_url', 'extra_info']
İstifadə olunan qiymət sütunu: price_x
Train dataseti həcmi: (317120, 55)
Test dataseti həcmi: (79280, 55)


## Checkpoint 2: Preprocessing Pipeline (Pipeline və ColumnTransformer)

Bu mərhələdə xüsusiyyətlərin model tərəfindən düzgün emal edilməsi üçün avtomatlaşdırılmış preprocessing iş axını qurdum.

Data Leakage riskini sıfıra endirmək və RAM yaddaşını optimallaşdırmaq üçün:
1. Rəqəmsal (numeric) sütunlarda çatışmayan dəyərləri median göstəricisi ilə doldurdum (SimpleImputer) və miqyas fərqini aradan qaldırmaq üçün standartlaşdırma tətbiq etdim (StandardScaler).
2. Kateqorial (categorical) sütunlarda çatışmayan dəyərləri mod göstəricisi ilə doldurdum və OneHotEncoder tətbiq etdim.
3. Bütün bu addımları ColumnTransformer daxilində birləşdirərək fit_transform əməliyyatını yalnız Train datasında icra etdim, Test datasını isə yalnız transform etdim.

In [8]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns if X_train[col].nunique() < 100]

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Hazırlanmış Train matrisinin ölçüsü:", X_train_prepared.shape)
print("Hazırlanmış Test matrisinin ölçüsü:", X_test_prepared.shape)

Hazırlanmış Train matrisinin ölçüsü: (317120, 431)
Hazırlanmış Test matrisinin ölçüsü: (79280, 431)


## Checkpoint 3: Ən Azı 2 Fərqli Modelin Öyrədilməsi

Avtomobil qiymətlərini proqnozlaşdırmaq (Reqressiya tapşırığı) üçün fərqli alqoritmik strukturlara malik 2 maşın öyrənməsi modelini öyrətdim:
1. **Linear Regression:** Baza (baseline) model kimi xüsusiyyətlər və qiymət arasındakı xətti asılılıqları qiymətləndirir.
2. **Random Forest Regressor:** Qeyri-xətti əlaqələri və xüsusiyyətlər arası mürəkkəb qarşılıqlı təsirləri tutmaq üçün tətbiq olunur.

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lr_model = LinearRegression()
lr_model.fit(X_train_prepared, y_train)

rf_model = RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_prepared, y_train)

print("Linear Regression və Random Forest modelləri uğurla öyrədildi.")

Linear Regression və Random Forest modelləri uğurla öyrədildi.


## Checkpoint 4: Modellərin Reqressiya Metrikləri ilə Qiymətləndirilməsi

Avtomobil qiymətlərinin proqnozlaşdırılması reqressiya məsələsi olduğu üçün modellərin fəaliyyətini 3 əsas metric üzrə qiymətləndirdim:
1. **MAE (Mean Absolute Error):** Modellərin ortalama neçə AZN xəta etdiyini mütləq dəyərlə göstərir.
2. **RMSE (Root Mean Squared Error):** Böyük xətalara (outlier-lərə) qarşı daha həssasdır və xətanın kvadrat kökünü hesablayır.
3. **R² Score (Determination Coefficient):** Modelin qiymətdəki dəyişkənliyin neçə faizini izah edə bildiyini göstərir (1-ə nə qədər yaxındırsa, bir o qədər yaxşıdır).

In [10]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_preds = lr_model.predict(X_test_prepared)
rf_preds = rf_model.predict(X_test_prepared)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print("Linear Regression Metrikləri:")
print("MAE:", round(lr_mae, 2), "AZN")
print("RMSE:", round(lr_rmse, 2), "AZN")
print("R2 Score:", round(lr_r2, 4))

print("\nRandom Forest Metrikləri:")
print("MAE:", round(rf_mae, 2), "AZN")
print("RMSE:", round(rf_rmse, 2), "AZN")
print("R2 Score:", round(rf_r2, 4))

Linear Regression Metrikləri:
MAE: 1.46 AZN
RMSE: 65.97 AZN
R2 Score: 1.0

Random Forest Metrikləri:
MAE: 10.03 AZN
RMSE: 94.59 AZN
R2 Score: 1.0


## Checkpoint: Cross-Validation Tətbiqi (5-Fold CV)

Modelin yalnız tək bir Train/Test bölgüsündən asılı olub təsadüfi uğur qazanmadığını yoxlamaq üçün **5-Fold Cross-Validation** (Çapraz Doğrulama) metodunu tətbiq etdim.

Məlumat sızmasının (Data Leakage) qarşısını almaq üçün Cross-Validation əməliyyatını yalnız öyrənmə datasında (`X_train_prepared`, `y_train`) icra etdim. Bu metod datanı 5 bərabər hissəyə bölür, hər dəfə 4 hissə ilə modeli öyrədir və 1 hissə ilə test edir.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('turbo.az_cars.csv', on_bad_lines='skip', engine='python')

target_col = [col for col in df.columns if 'price' in col.lower() or 'qiymət' in col.lower() or 'qiymet' in col.lower()][0]

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns if X_train[col].nunique() < 15]

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_prepared = preprocessor.fit_transform(X_train)

lr_model = LinearRegression()
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)

lr_cv_scores = cross_val_score(lr_model, X_train_prepared, y_train, cv=5, scoring='r2', n_jobs=-1)
dt_cv_scores = cross_val_score(dt_model, X_train_prepared, y_train, cv=5, scoring='r2', n_jobs=-1)

print("Linear Regression 5-Fold R2 Skorları:", lr_cv_scores)
print("Linear Regression Ortalama R2 Skoru:", round(lr_cv_scores.mean(), 4))

print("\nDecision Tree 5-Fold R2 Skorları:", dt_cv_scores)
print("Decision Tree Ortalama R2 Skoru:", round(dt_cv_scores.mean(), 4))

Linear Regression 5-Fold R2 Skorları: [0.99999754 0.99999651 0.99998918 0.9999965  0.99999772]
Linear Regression Ortalama R2 Skoru: 1.0

Decision Tree 5-Fold R2 Skorları: [0.99997306 0.9828888  0.99998603 0.99998366 0.9955378 ]
Decision Tree Ortalama R2 Skoru: 0.9957


## Feature Importance Analizi və İnterpretasiyası

Bu mərhələdə avtomobil qiymətinin proqnozlaşdırılmasında hansı xüsusiyyətlərin (features) daha böyük rola malik olduğunu təyin etmək üçün Decision Tree modelinin `feature_importances_` göstəricilərini analiz etdim.

In [2]:
import pandas as pd
import numpy as np

dt_model.fit(X_train_prepared, y_train)

feature_names = preprocessor.get_feature_names_out()
importances = dt_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 10 Ən Əhəmiyyətli Xüsusiyyət (Feature Importance):")
print(importance_df.head(10))

Top 10 Ən Əhəmiyyətli Xüsusiyyət (Feature Importance):
                                      Feature    Importance
3                                num__price_y  9.999974e-01
4                                  num__phone  1.165919e-06
45                             cat__Yeni_Bəli  3.837380e-07
1                num__engine_displacement_num  2.789097e-07
6                          num__Buraxılış ili  2.610183e-07
5                                  num__views  1.226509e-07
50                       cat__Yerlərin sayı_4  1.067236e-07
0                        num__production_year  6.690719e-08
7                           cat__currency_x_$  5.336190e-08
27  cat__Hansı bazar üçün yığılıb_Rəsmi diler  5.060964e-08
